# Bottom-Up Heatmap HPE (Scratch, Fast Kaggle)

Sumbu A: bottom-up

Sumbu B: heatmap

Baseline ini memakai full-image heatmap untuk keypoint + center map (grouping sederhana).

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from hpe_shared import (
    benchmark_latency,
    compute_oks_pck,
    decode_heatmaps_argmax,
    gaussian_heatmap,
    get_device,
    load_coco_keypoint_samples,
    read_image_bgr,
    seed_everything,
    split_samples,
    to_tensor_rgb,
    vis_mask_from_kpts,
)

seed_everything(42)
device = get_device()
print('device:', device)

In [ ]:
DATA_ROOT = Path('/kaggle/input/datasets/yanplayz08/coco-subset-for-pose-estimation')
ANN_PATH = DATA_ROOT / 'annotations' / 'person_keypoints_train2017.json'
IMG_DIR = DATA_ROOT / 'train2017'

MAX_SAMPLES = 3000
TRAIN_BS = 32
EPOCHS = 3
LR = 1e-3
IN_SIZE = 256
HM_SIZE = 64

samples = load_coco_keypoint_samples(str(ANN_PATH), str(IMG_DIR), max_samples=MAX_SAMPLES, min_labeled_kpt=5)
train_samples, val_samples = split_samples(samples, val_ratio=0.2, seed=42)
print('samples:', len(samples), 'train:', len(train_samples), 'val:', len(val_samples))

In [ ]:
class BottomUpDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = read_image_bgr(s.image_path)
        h0, w0 = img.shape[:2]
        x = to_tensor_rgb(img, (IN_SIZE, IN_SIZE)).float()

        kpts = s.keypoints.copy()
        kpts[:, 0] = np.clip(kpts[:, 0] / max(w0, 1), 0.0, 1.0)
        kpts[:, 1] = np.clip(kpts[:, 1] / max(h0, 1), 0.0, 1.0)
        vis = vis_mask_from_kpts(kpts)

        kpt_hm = np.zeros((17, HM_SIZE, HM_SIZE), dtype=np.float32)
        for j in range(17):
            if vis[j] <= 0:
                continue
            cx = float(kpts[j, 0] * (HM_SIZE - 1))
            cy = float(kpts[j, 1] * (HM_SIZE - 1))
            kpt_hm[j] = gaussian_heatmap(HM_SIZE, HM_SIZE, cx, cy, sigma=1.8)

        center = np.zeros((1, HM_SIZE, HM_SIZE), dtype=np.float32)
        vis_idx = np.where(vis > 0)[0]
        if len(vis_idx) > 0:
            cx = float(np.mean(kpts[vis_idx, 0]) * (HM_SIZE - 1))
            cy = float(np.mean(kpts[vis_idx, 1]) * (HM_SIZE - 1))
            center[0] = gaussian_heatmap(HM_SIZE, HM_SIZE, cx, cy, sigma=2.2)

        target = np.concatenate([kpt_hm, center], axis=0)

        return {
            'image': x,
            'target': torch.from_numpy(target),
            'gt_xy_norm': torch.from_numpy(kpts[:, :2].astype(np.float32)),
            'gt_vis': torch.from_numpy(vis.astype(np.float32)),
            'area': torch.tensor(float(max(1.0, s.area)), dtype=torch.float32),
        }


class TinyBottomUp(nn.Module):
    def __init__(self, out_ch=18):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, out_ch, 1),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
train_dl = DataLoader(BottomUpDataset(train_samples), batch_size=TRAIN_BS, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(BottomUpDataset(val_samples), batch_size=TRAIN_BS, shuffle=False, num_workers=2, pin_memory=True)

model = TinyBottomUp().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for b in train_dl:
        x = b['image'].to(device)
        y = b['target'].to(device)

        pred = model(x)
        loss = F.mse_loss(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()
        running += float(loss.item())

    print(f'epoch={epoch+1} train_loss={running/max(1,len(train_dl)):.5f}')

In [ ]:
model.eval()
oks_all, pck_all, miss_all = [], [], []

with torch.no_grad():
    for b in val_dl:
        x = b['image'].to(device)
        pred = model(x)
        pred_kpt_hm = pred[:, :17, :, :]

        pred_xy_norm = decode_heatmaps_argmax(pred_kpt_hm).cpu().numpy()
        gt_xy_norm = b['gt_xy_norm'].numpy()
        gt_vis = b['gt_vis'].numpy()
        area = b['area'].numpy()

        for i in range(pred_xy_norm.shape[0]):
            pred_xy_px = np.zeros((17, 2), dtype=np.float32)
            pred_xy_px[:, 0] = pred_xy_norm[i, :, 0] * IN_SIZE
            pred_xy_px[:, 1] = pred_xy_norm[i, :, 1] * IN_SIZE

            gt_kpts = np.zeros((17, 3), dtype=np.float32)
            gt_kpts[:, 0] = gt_xy_norm[i, :, 0] * IN_SIZE
            gt_kpts[:, 1] = gt_xy_norm[i, :, 1] * IN_SIZE
            gt_kpts[:, 2] = gt_vis[i]

            pred_vis = np.ones((17,), dtype=np.float32)
            oks, pck, miss = compute_oks_pck(pred_xy_px, pred_vis, gt_kpts, area=float(area[i]), pck_alpha=0.2)
            oks_all.append(oks)
            pck_all.append(pck)
            miss_all.append(miss)

lat_in = torch.randn(1, 3, IN_SIZE, IN_SIZE, device=device)
lat_ms, fps = benchmark_latency(model, lat_in, iters=120)

print('Bottom-Up Heatmap Summary')
print('OKS mean           :', float(np.mean(oks_all) if oks_all else 0.0))
print('PCK mean           :', float(np.mean(pck_all) if pck_all else 0.0))
print('Missing joint ratio:', float(np.mean(miss_all) if miss_all else 0.0))
print('Latency mean (ms)  :', float(lat_ms))
print('FPS approx         :', float(fps))